# AETHER 2.0 — Análise de Resultados e Escala de Decisão
**FIAP Global Solution 2026 · Dynamic Programming**

Notebook de análise interativa dos algoritmos de monitoramento de risco ambiental.
Cenário: enchentes no Rio Grande do Sul (2024).

## 1. Carga de dados reais e construção das estruturas

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from src.data_structures import carregar_municipios, construir_grafo, construir_bst
from src.greedy import dijkstra, reconstruir_caminho, rotas_a_partir_do_hub
from src.brute_force import forca_bruta_caminho_minimo
from src.performance_monitor import comparar_algoritmos, crescimento_caminhos, imprimir_tabela

munis = carregar_municipios('../data/raw/municipios_rs_2024.csv')
g = construir_grafo(munis)
bst = construir_bst(munis)
print(f'Municipios: {len(munis)}')
print(f'Grafo: {g.num_vertices()} vertices, {g.num_arestas()} arestas, conexo={g.is_conexo()}')
print(f'BST: altura={bst.altura()}, balanceada={bst.esta_balanceada()}')

Municipios: 30
Grafo: 30 vertices, 220 arestas, conexo=True
BST: altura=5, balanceada=True


## 2. Priorização por risco (BST in-order decrescente)

In [2]:
print('TOP 10 municipios mais criticos:')
for i, m in enumerate(bst.percurso_in_order_desc()[:10], 1):
    print(f'{i:2d}. {m.nome:25s} risco={m.indice_risco:.2f} pop={m.populacao:,}')

TOP 10 municipios mais criticos:
 1. Muçum                     risco=0.98 pop=4,669
 2. Roca Sales                risco=0.96 pop=11,952
 3. São Sebastião do Caí      risco=0.95 pop=24,716
 4. Arroio do Meio            risco=0.94 pop=20,226
 5. Estrela                   risco=0.93 pop=33,126
 6. Canoas                    risco=0.92 pop=347,801
 7. Encantado                 risco=0.91 pop=23,085
 8. Taquari                   risco=0.90 pop=30,343
 9. Rolante                   risco=0.89 pop=21,238
10. Lajeado                   risco=0.88 pop=95,124


## 3. Rotas otimas de atendimento (Dijkstra a partir de Porto Alegre)

In [3]:
HUB = 4314902  # Porto Alegre
rotas = rotas_a_partir_do_hub(g, HUB)
criticos = bst.percurso_in_order_desc()[:8]
for m in criticos:
    if m.id_ibge == HUB:
        continue
    r = rotas[m.id_ibge]
    print(f'{m.nome:22s} custo={r["custo_km"]:6.1f}km  rota: {" -> ".join(r["caminho_nomes"])}')

Muçum                  custo= 115.4km  rota: Porto Alegre -> Montenegro -> Muçum
Roca Sales             custo= 104.5km  rota: Porto Alegre -> Montenegro -> Roca Sales
São Sebastião do Caí   custo=  52.0km  rota: Porto Alegre -> São Sebastião do Caí
Arroio do Meio         custo= 102.2km  rota: Porto Alegre -> Montenegro -> Arroio do Meio
Estrela                custo=  98.2km  rota: Porto Alegre -> Montenegro -> Estrela
Canoas                 custo=  13.4km  rota: Porto Alegre -> Canoas
Encantado              custo= 109.1km  rota: Porto Alegre -> Montenegro -> Encantado
Taquari                custo=  67.6km  rota: Porto Alegre -> Taquari


## 4. Comparacao Forca Bruta x Dijkstra

In [4]:
registros = comparar_algoritmos()
imprimir_tabela(registros)


   N | Arestas | Dijkstra(ms) |       FB(ms) |  Dij ops |     FB ops |  Gap %
------------------------------------------------------------------------------
   5 |       5 |       0.0449 |       0.0439 |       10 |          6 |   0.00
   8 |       7 |       0.0369 |       0.0339 |       15 |          8 |   0.00
  10 |      16 |       0.0427 |       0.1063 |       26 |         53 |   0.00
  12 |      31 |       0.1047 |       0.1375 |       47 |         90 |   0.00
  20 |      91 |       0.0871 |     inviável |      119 |          - |      -
  50 |     560 |       0.5937 |     inviável |      651 |          - |      -
 100 |    2174 |       3.2171 |     inviável |    2,389 |          - |      -


## 5. Explosao combinatoria da Forca Bruta

In [5]:
print('N -> numero de caminhos simples:')
for n, total in crescimento_caminhos():
    print(f'  N={n:>3} -> {total:>14,} caminhos')

N -> numero de caminhos simples:


  N=  4 ->              5 caminhos
  N=  6 ->             49 caminhos
  N=  8 ->            587 caminhos
  N= 10 ->         31,190 caminhos
  N= 12 ->      1,029,308 caminhos


## 6. Escala de Decisao (4 niveis)

Classificacao das solucoes considerando o trade-off entre qualidade e custo computacional:

| Nivel | Estrategia | Quando usar | Qualidade | Custo |
|---|---|---|---|---|
| **1 - Otimo garantido** | Forca Bruta | N <= 12 (validacao) | 100% (otimo) | Exponencial - inviavel |
| **2 - Otimo eficiente** | Dijkstra (guloso) | Qualquer N com pesos >= 0 | 100% (otimo, gap 0%) | O((V+E) log V) |
| **3 - Cobertura** | Prim/Kruskal (MST) | Posicionar bases de recursos | Otimo p/ cobertura | O(E log V) |
| **4 - Heuristico** | A* / gulosos aprox. | N massivo, tempo real | Aproximado | Sub-linear no pior caso |

**Recomendacao AETHER:** usar **Dijkstra (Nivel 2)** como motor de despacho.

## 7. Conclusao e Conexao com os ODS

- **ODS 9** (Infraestrutura e Inovacao): algoritmos eficientes viabilizam resposta logistica.
- **ODS 11** (Cidades Sustentaveis): protecao de populacoes urbanas em desastres.
- **ODS 13** (Acao Climatica): resposta rapida a eventos climaticos extremos.

O Dijkstra atinge o mesmo resultado da Forca Bruta (gap 0%) com custo viavel.